# Merge Sentinel-1 RTC extraction shards and build yearly outputs

This notebook is the **finalization step only**. It does not query Planetary Computer and does not download/extract scenes.

It will:

1. Use the Azure project copied to `~/sentinel1-rtc` as the destination.
2. Merge any separate WSL `scene_observations_nearest_overview_v1/*.npz` shards into that destination, copying only absent shards. Same-name duplicates are compared by scene day, station IDs, and array values; conflicting copies stop the notebook rather than overwrite data.
3. Validate the saved `scene_inventory_nearest_v1.sqlite`, `stations_with_clusters.parquet`, and all expected scene shards.
4. Produce/resume `nearest_valid_overview_v1/samples_nearest_valid_YEAR.parquet` and `s1_nearest_valid_YEAR.zarr` for every year in the source sample table.

## Before running

- Open this notebook with the project root as `~/sentinel1-rtc`, or adjust `PROJECT_ROOT` in the configuration cell.
- This is preconfigured for the project tree you showed: WSL source shards at `~/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1` and Azure destination shards at `~/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1`.
- The notebook merges only shard files. It uses the Azure copy's inventory, station-cluster mapping, stations, and sample-date table for validation and yearly aggregation. If the WSL run used a different inventory or station/sample inputs, do not bypass validation; those inputs need to be reconciled first.
- If the WSL shard folder has a different location, edit `WSL_SHARD_DIRS` in the configuration cell before running.
- Run the cells from top to bottom. The Zarr writer checkpoints `sample_done`, so rerunning after a disk-space pause resumes completed sample ranges.

The outputs preserve the source notebook's `overview` method and VV/VH-only extraction layout. HH/HV arrays remain empty; this notebook does not invent those observations.

In [1]:
# ============================================================
# CONFIGURATION
# ============================================================
from pathlib import Path
import os
import re
import gc
import csv
import json
import time
import shutil
import sqlite3
import hashlib
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import zarr
    from zarr.codecs import BloscCodec, BloscShuffle
except Exception as exc:
    raise RuntimeError(
        "This notebook expects the Zarr v3 API used by the extraction notebooks. "
        "Install/activate the same Python environment used for BUILD_SENTINEL_RTC_GLOBAL_v12_EIGHT_SCENE_WORKERS."
    ) from exc

# Azure working directory copied to the WSL home directory.
PROJECT_ROOT = Path.home() / "sentinel1-rtc"
# If you opened the notebook from inside a different project directory, this fallback
# looks at the current directory and its parents for the expected Data tree.
_EXPECTED_SENTINEL_REL = Path("Data/Global/pipeline_stage1/sentinel1")
if not (PROJECT_ROOT / _EXPECTED_SENTINEL_REL).is_dir():
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / _EXPECTED_SENTINEL_REL).is_dir():
            PROJECT_ROOT = _candidate.resolve()
            break

DATA_DIR = PROJECT_ROOT / "Data/Global/pipeline_stage1"
OUTPUT_DIR = DATA_DIR / "sentinel1"
SCENE_OBS_DIR = OUTPUT_DIR / "scene_observations_nearest_overview_v1"
SCENE_INDEX_DB_PATH = OUTPUT_DIR / "scene_inventory_nearest_v1.sqlite"
STATIONS_PATH = DATA_DIR / "stations.parquet"
SAMPLES_PATH = DATA_DIR / "station_date_candidates.parquet"
STATION_CLUSTER_MAP_PATH = OUTPUT_DIR / "stations_with_clusters.parquet"
FINAL_OUTPUT_DIR = OUTPUT_DIR / "nearest_valid_overview_v1"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Based on the supplied WSL tree, the local run lives directly under ~/Data,
# while the copied Azure project (destination) lives under ~/sentinel1-rtc/Data.
WSL_SHARD_DIRS = [
    Path.home() / "Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1"
]

# Optional additional project roots can be listed here if there are more WSL runs.
WSL_PROJECT_ROOTS = []
AUTO_DISCOVER_SIBLING_SHARD_DIRS = True

# Matches both supplied extraction notebooks: overview values, 4 stored real bands, 13x13 patches.
PRODUCTION_READ_STRATEGY = "overview"
SCHEMA_VERSION = 6
S1_COLLECTION = "sentinel-1-rtc"
PATCH_SIZE = 13
MODEL_RESOLUTION_M = 1000
S1_RESOLUTION_M = 10
MAX_NEAREST_DAYS = 18
MIN_VALID_PAIR_PIXELS = 1
REAL_BANDS = ["vv", "vh", "hh", "hv"]
DERIVED_BANDS = ["vv_vh_ratio"]
OUTPUT_BANDS = REAL_BANDS + DERIVED_BANDS
N_REAL_BANDS = len(REAL_BANDS)
N_OUTPUT_BANDS = len(OUTPUT_BANDS)
SOURCE_OFFSET_MISSING = int(np.iinfo(np.int16).max)
READ_POLARIZATIONS = ("vv", "vh")

# The Azure extraction notebook explicitly permits this one absent scene.
# Every other inventory scene is required before aggregation starts.
OPTIONAL_MISSING_SCENE_IDS = {
    "S1B_IW_GRDH_1SDV_20190120T064614_20190120T064639_014574_01B268_rtc",
}

# Preserve the source pipeline's low-disk safeguards.
WARN_FREE_GIB = 10.0
STOP_FREE_GIB = 8.0
RUN_YEARLY_AGGREGATION = True

REPORT_DIR = OUTPUT_DIR / "merge_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MERGE_REPORT_PATH = REPORT_DIR / "wsl_azure_shard_merge_report.csv"
MISSING_SHARDS_REPORT_PATH = REPORT_DIR / "missing_scene_shards.csv"

print("Project root:", PROJECT_ROOT)
print("Destination shard directory:", SCENE_OBS_DIR)
print("Inventory database:", SCENE_INDEX_DB_PATH)
print("Station-cluster map:", STATION_CLUSTER_MAP_PATH)
print("Yearly output directory:", FINAL_OUTPUT_DIR)
assert PROJECT_ROOT.is_dir(), f"Project root does not exist: {PROJECT_ROOT}"
assert OUTPUT_DIR.is_dir(), f"Sentinel-1 output directory does not exist: {OUTPUT_DIR}"
assert SCENE_OBS_DIR.is_dir(), f"Scene shard directory does not exist: {SCENE_OBS_DIR}"
assert SCENE_INDEX_DB_PATH.is_file(), f"Inventory database not found: {SCENE_INDEX_DB_PATH}"
assert STATIONS_PATH.is_file(), f"Stations parquet not found: {STATIONS_PATH}"
assert SAMPLES_PATH.is_file(), f"Samples parquet not found: {SAMPLES_PATH}"
assert STATION_CLUSTER_MAP_PATH.is_file(), f"Saved station-cluster map not found: {STATION_CLUSTER_MAP_PATH}"


Project root: /home/mayur/Code/UG_SM/sentinel1-rtc
Destination shard directory: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Inventory database: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
Station-cluster map: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/stations_with_clusters.parquet
Yearly output directory: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1


In [2]:
# ============================================================
# LOCATE SOURCE SHARD DIRECTORIES
# ============================================================
def _as_path(value):
    return Path(value).expanduser().resolve()


def discover_other_shard_dirs():
    found = set()
    for value in WSL_SHARD_DIRS:
        p = _as_path(value)
        if p.is_dir():
            found.add(p)
        else:
            print("Configured WSL shard path does not exist (ignored):", p)

    for value in WSL_PROJECT_ROOTS:
        p = _as_path(value) / _EXPECTED_SENTINEL_REL / "scene_observations_nearest_overview_v1"
        if p.is_dir():
            found.add(p.resolve())
        else:
            print("Configured WSL project root has no overview shard folder (ignored):", p)

    if AUTO_DISCOVER_SIBLING_SHARD_DIRS:
        # Search common project-root locations only; avoid an unbounded scan of the whole filesystem.
        search_bases = {
            Path.home().resolve(),
            PROJECT_ROOT.parent.resolve(),
            Path.cwd().resolve(),
        }
        for base in search_bases:
            patterns = [
                "*/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1",
                "*/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1",
            ]
            for pattern in patterns:
                try:
                    found.update(p.resolve() for p in base.glob(pattern) if p.is_dir())
                except OSError:
                    pass

    destination = SCENE_OBS_DIR.resolve()
    return sorted((p for p in found if p != destination), key=str)


SOURCE_SHARD_DIRS = discover_other_shard_dirs()
print("Destination (will receive missing shards):", SCENE_OBS_DIR.resolve())
if SOURCE_SHARD_DIRS:
    print("Additional WSL/source shard folders:")
    for p in SOURCE_SHARD_DIRS:
        print(" -", p, f"({sum(1 for _ in p.glob('*.npz')):,} NPZ files)")
else:
    print("No separate WSL shard folder found. This is OK if the directory copy already merged both sets.")
print("Current destination NPZ files:", f"{sum(1 for _ in SCENE_OBS_DIR.glob('*.npz')):,}")


Configured WSL shard path does not exist (ignored): /home/mayur/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Destination (will receive missing shards): /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
No separate WSL shard folder found. This is OK if the directory copy already merged both sets.
Current destination NPZ files: 204,032


In [3]:
# ============================================================
# SAFE, RESUMABLE SHARD MERGE
# ============================================================
def shard_path_for_scene(scene_id, folder=SCENE_OBS_DIR):
    scene_id = str(scene_id)
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scene_id)
    suffix = hashlib.sha1(scene_id.encode("utf-8")).hexdigest()[:10]
    return Path(folder) / f"{safe[:140]}_{suffix}.npz"


def _validate_shard_npz_container(path):
    """Validate ZIP/NPZ structure and required keys without decompressing raster arrays."""
    if not zipfile.is_zipfile(path):
        raise ValueError(f"Not a valid NPZ ZIP container: {path}")
    with np.load(path, allow_pickle=False) as z:
        required = {"station_ids", "day", "data"}
        missing = required - set(z.files)
        if missing:
            raise ValueError(f"{path} is missing NPZ keys: {sorted(missing)}")


def _read_shard_payload(path):
    with np.load(path, allow_pickle=False) as z:
        required = {"station_ids", "day", "data"}
        missing = required - set(z.files)
        if missing:
            raise ValueError(f"{path} is missing NPZ keys: {sorted(missing)}")
        station_ids = z["station_ids"].astype(str)
        day = int(z["day"])
        data = z["data"].astype(np.float32, copy=False)
    if data.ndim != 4 or data.shape[0] != len(station_ids) or data.shape[1:] != (N_REAL_BANDS, PATCH_SIZE, PATCH_SIZE):
        raise ValueError(
            f"Invalid shard schema in {path}: station_ids={station_ids.shape}, data={data.shape}; "
            f"expected (n, {N_REAL_BANDS}, {PATCH_SIZE}, {PATCH_SIZE})."
        )
    if len(set(station_ids.tolist())) != len(station_ids):
        raise ValueError(f"Duplicate station IDs inside shard: {path}")
    return station_ids, day, data


def _shards_semantically_equal(path_a, path_b):
    ids_a, day_a, data_a = _read_shard_payload(path_a)
    ids_b, day_b, data_b = _read_shard_payload(path_b)
    if day_a != day_b or set(ids_a.tolist()) != set(ids_b.tolist()):
        return False
    if data_a.shape[1:] != data_b.shape[1:]:
        return False
    pos_b = {sid: i for i, sid in enumerate(ids_b.tolist())}
    aligned_b = data_b[[pos_b[sid] for sid in ids_a.tolist()]]
    return bool(np.array_equal(data_a, aligned_b, equal_nan=True))


def merge_one_source(source_dir):
    source_dir = Path(source_dir).resolve()
    candidates = sorted(source_dir.glob("*.npz"))
    new_count = 0
    same_count = 0
    conflict_count = 0
    bad_count = 0
    rows = []
    conflicts = []

    for src in tqdm(candidates, desc=f"Merging {source_dir.name}", unit="shard"):
        dst = SCENE_OBS_DIR / src.name
        try:
            # Key/container check does not decompress the large data arrays.
            _validate_shard_npz_container(src)
        except Exception as exc:
            bad_count += 1
            rows.append({"source_dir": str(source_dir), "filename": src.name, "status": f"invalid_source:{type(exc).__name__}:{exc}", "source_bytes": src.stat().st_size, "destination_bytes": None})
            continue

        if not dst.exists():
            temp_dst = dst.with_name(dst.name + ".merge_tmp")
            try:
                shutil.copy2(src, temp_dst)
                _validate_shard_npz_container(temp_dst)
                os.replace(temp_dst, dst)
                new_count += 1
                rows.append({"source_dir": str(source_dir), "filename": src.name, "status": "copied_new", "source_bytes": src.stat().st_size, "destination_bytes": dst.stat().st_size})
            except Exception as exc:
                try:
                    temp_dst.unlink(missing_ok=True)
                except Exception:
                    pass
                bad_count += 1
                rows.append({"source_dir": str(source_dir), "filename": src.name, "status": f"copy_failed:{type(exc).__name__}:{exc}", "source_bytes": src.stat().st_size, "destination_bytes": None})
            continue

        try:
            equal = _shards_semantically_equal(src, dst)
        except Exception as exc:
            equal = False
            detail = f"comparison_failed:{type(exc).__name__}:{exc}"
        else:
            detail = "same_scene_payload" if equal else "payload_mismatch"

        if equal:
            same_count += 1
            rows.append({"source_dir": str(source_dir), "filename": src.name, "status": "duplicate_identical_skipped", "source_bytes": src.stat().st_size, "destination_bytes": dst.stat().st_size})
        else:
            conflict_count += 1
            conflicts.append({"source_dir": str(source_dir), "filename": src.name, "status": detail, "source_bytes": src.stat().st_size, "destination_bytes": dst.stat().st_size})
            rows.append(conflicts[-1])

    return {
        "source_dir": str(source_dir), "source_npz": len(candidates),
        "new_copied": new_count, "identical_duplicates": same_count,
        "conflicts": conflict_count, "invalid_or_failed": bad_count,
        "rows": rows, "conflict_rows": conflicts,
    }


merge_results = []
for source_dir in SOURCE_SHARD_DIRS:
    merge_results.append(merge_one_source(source_dir))

merge_rows = [row for result in merge_results for row in result["rows"]]
if not merge_rows:
    merge_rows = [{"source_dir": "", "filename": "", "status": "no_external_sources_to_merge", "source_bytes": None, "destination_bytes": None}]
pd.DataFrame(merge_rows).to_csv(MERGE_REPORT_PATH, index=False)

merge_summary = pd.DataFrame([
    {k: v for k, v in result.items() if k not in {"rows", "conflict_rows"}}
    for result in merge_results
])
if not merge_summary.empty:
    display(merge_summary)
else:
    print("No separate source directory needed; existing destination is treated as the merged shard collection.")
print("Merge report:", MERGE_REPORT_PATH)

conflicts = [row for result in merge_results for row in result["conflict_rows"]]
failures = [
    row for result in merge_results for row in result["rows"]
    if str(row["status"]).startswith(("invalid_source", "copy_failed"))
]
if conflicts:
    print("Conflicting duplicate shards (first 20):")
    display(pd.DataFrame(conflicts).head(20))
    raise RuntimeError(
        f"Found {len(conflicts):,} same-name shard conflicts. Nothing was overwritten; "
        f"inspect {MERGE_REPORT_PATH} before proceeding. Yearly aggregation is blocked."
    )
if failures:
    display(pd.DataFrame(failures).head(20))
    raise RuntimeError(
        f"Found {len(failures):,} invalid/failed source shards. Inspect {MERGE_REPORT_PATH}; "
        "yearly aggregation is blocked."
    )

print("Destination NPZ files after merge:", f"{sum(1 for _ in SCENE_OBS_DIR.glob('*.npz')):,}")


No separate source directory needed; existing destination is treated as the merged shard collection.
Merge report: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/merge_reports/wsl_azure_shard_merge_report.csv
Destination NPZ files after merge: 204,032


In [4]:
# ============================================================
# LOAD THE SAVED INPUTS AND VALIDATE THE ORIGINAL CLUSTER MAPPING
# ============================================================
def _load_station_sample_inputs():
    # Inspect schema first so a wide sample parquet is never fully materialized.
    try:
        import pyarrow.parquet as _pq
        sample_available_columns = set(_pq.ParquetFile(SAMPLES_PATH).schema.names)
    except ImportError:
        try:
            import fastparquet as _fp
            sample_available_columns = set(_fp.ParquetFile(SAMPLES_PATH).columns)
        except ImportError:
            # Conservative fallback: project the required columns only. sample_id is optional.
            sample_available_columns = {"station_id", "sample_date"}
    keep = [c for c in ("sample_id", "station_id", "sample_date") if c in sample_available_columns]
    missing = {"station_id", "sample_date"} - set(keep)
    if missing:
        raise KeyError(f"{SAMPLES_PATH} is missing required sample columns: {sorted(missing)}")
    samples = pd.read_parquet(SAMPLES_PATH, columns=keep)

    stations = pd.read_parquet(STATIONS_PATH, columns=["station_id", "latitude", "longitude"])
    mapping = pd.read_parquet(STATION_CLUSTER_MAP_PATH)
    if not {"station_id", "cluster_id"}.issubset(mapping.columns):
        raise KeyError(f"Saved mapping must contain station_id and cluster_id: {list(mapping.columns)}")
    mapping = mapping[["station_id", "cluster_id"]].copy()

    samples["station_id"] = samples["station_id"].astype(str)
    stations["station_id"] = stations["station_id"].astype(str)
    mapping["station_id"] = mapping["station_id"].astype(str)
    mapping["cluster_id"] = pd.to_numeric(mapping["cluster_id"], errors="raise").astype(np.int64)
    samples["sample_date"] = pd.to_datetime(samples["sample_date"], errors="coerce").dt.normalize()

    stations = stations.dropna(subset=["latitude", "longitude"])
    stations = stations.loc[
        stations["latitude"].between(-90, 90) & stations["longitude"].between(-180, 180)
    ].drop_duplicates("station_id").reset_index(drop=True)
    samples = samples.dropna(subset=["sample_date"])
    samples = samples.loc[samples["station_id"].isin(stations["station_id"])].copy()
    stations = stations.loc[stations["station_id"].isin(samples["station_id"].unique())].reset_index(drop=True)

    if mapping["station_id"].duplicated().any():
        raise RuntimeError("Saved station-cluster mapping has duplicate station IDs.")
    current_ids = set(stations["station_id"])
    saved_ids = set(mapping["station_id"])
    if current_ids != saved_ids:
        raise RuntimeError(
            "Saved station-cluster mapping does not match the cleaned stations input "
            f"(missing={len(current_ids-saved_ids):,}, extra={len(saved_ids-current_ids):,})."
        )

    lookup = mapping.set_index("station_id")["cluster_id"]
    sample_clusters = samples["station_id"].map(lookup)
    if sample_clusters.isna().any():
        raise RuntimeError("Some sample stations have no saved cluster assignment.")
    samples["cluster_id"] = sample_clusters.astype("int32")
    samples["year"] = samples["sample_date"].dt.year.astype("int16")
    if samples.empty:
        raise RuntimeError("No valid station-date samples remain after input validation.")
    if samples["cluster_id"].isna().any() or (samples["cluster_id"] < 0).any():
        raise RuntimeError("Invalid cluster IDs in samples.")
    return samples, stations, mapping


samples_c, stations_clean, station_clusters = _load_station_sample_inputs()
print("Validated stations:", f"{stations_clean['station_id'].nunique():,}")
print("Validated sample rows:", f"{len(samples_c):,}")
print("Saved clusters:", f"{station_clusters['cluster_id'].nunique():,}")
print("Sample years:", sorted(samples_c["year"].unique().astype(int).tolist()))


Validated stations: 2,210
Validated sample rows: 3,514,643
Saved clusters: 819
Sample years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [5]:
# ============================================================
# VALIDATE SQLITE INVENTORY COMPLETENESS AND SHARD COVERAGE
# ============================================================
def validate_inventory_and_shards():
    conn = sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120)
    try:
        table_names = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        required_tables = {"scenes", "scene_clusters", "cluster_searches"}
        missing_tables = required_tables - table_names
        if missing_tables:
            raise RuntimeError(f"Inventory DB lacks required tables: {sorted(missing_tables)}")

        n_scenes = int(conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
        n_links = int(conn.execute("SELECT COUNT(*) FROM scene_clusters").fetchone()[0])
        scenes_without_links = int(conn.execute(
            """SELECT COUNT(*) FROM scenes s LEFT JOIN scene_clusters sc ON sc.scene_id=s.scene_id
               WHERE sc.scene_id IS NULL"""
        ).fetchone()[0])
        expected_clusters = set(station_clusters["cluster_id"].astype(int).unique())
        recorded_clusters = {int(r[0]) for r in conn.execute("SELECT DISTINCT cluster_id FROM cluster_searches")}
        completed_clusters = {int(r[0]) for r in conn.execute("SELECT cluster_id FROM cluster_searches WHERE completed=1")}
        inventory_scenes = conn.execute(
            """SELECT DISTINCT s.scene_id, s.day FROM scenes s
               JOIN scene_clusters sc ON sc.scene_id=s.scene_id
               ORDER BY s.day, s.scene_id"""
        ).fetchall()
    finally:
        conn.close()

    if scenes_without_links:
        raise RuntimeError(f"Inventory has {scenes_without_links:,} scenes with no cluster links.")
    if recorded_clusters != expected_clusters:
        raise RuntimeError(
            "SQLite cluster IDs differ from the saved station mapping. "
            f"Expected {len(expected_clusters):,}; recorded {len(recorded_clusters):,}. "
            "Do not aggregate against a mismatched inventory."
        )
    incomplete = expected_clusters - completed_clusters
    if incomplete:
        raise RuntimeError(
            f"Inventory contains {len(incomplete):,} incomplete cluster searches; "
            f"first IDs: {sorted(incomplete)[:20]}. Aggregation would risk missing nearer scenes."
        )

    missing_rows = []
    present_count = 0
    optional_count = 0
    # File existence check is much cheaper than opening/decompressing every NPZ at this stage.
    for scene_id, day in tqdm(inventory_scenes, desc="Checking inventory shard coverage", unit="scene"):
        scene_id = str(scene_id)
        path = shard_path_for_scene(scene_id)
        if path.is_file():
            present_count += 1
        elif scene_id in OPTIONAL_MISSING_SCENE_IDS:
            optional_count += 1
        else:
            missing_rows.append({"scene_id": scene_id, "expected_day": int(day), "expected_filename": path.name})

    pd.DataFrame(missing_rows, columns=["scene_id", "expected_day", "expected_filename"]).to_csv(
        MISSING_SHARDS_REPORT_PATH, index=False
    )
    summary = {
        "inventory_scenes_with_cluster_links": len(inventory_scenes),
        "scene_cluster_links": n_links,
        "shards_present": present_count,
        "explicitly_optional_missing": optional_count,
        "unexpected_missing_shards": len(missing_rows),
        "completed_cluster_searches": len(completed_clusters),
        "expected_cluster_searches": len(expected_clusters),
    }
    display(pd.DataFrame([summary]))
    print("Missing-shard report:", MISSING_SHARDS_REPORT_PATH)
    if missing_rows:
        display(pd.DataFrame(missing_rows).head(30))
        raise FileNotFoundError(
            f"{len(missing_rows):,} required scene shards are missing. "
            "Add the WSL shard folder to WSL_SHARD_DIRS/WSL_PROJECT_ROOTS (or copy its NPZ files into the destination), "
            f"then rerun the merge and coverage cells. See {MISSING_SHARDS_REPORT_PATH}. "
            "Yearly aggregation has not started."
        )
    print("Inventory and shard coverage validation: PASS")
    return summary


coverage_summary = validate_inventory_and_shards()


Checking inventory shard coverage:   0%|          | 0/204033 [00:00<?, ?scene/s]

,inventory_scenes_with_cluster_links,scene_cluster_links,shards_present,explicitly_optional_missing,unexpected_missing_shards,completed_cluster_searches,expected_cluster_searches
0,204033,528280,204032,1,0,819,819


Missing-shard report: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/merge_reports/missing_scene_shards.csv
Inventory and shard coverage validation: PASS


In [6]:
# ============================================================
# SCENE LOAD + NEAREST-VALID COMPILER
# ============================================================
def load_scene_observation(scene_id):
    path = shard_path_for_scene(scene_id)
    if not path.is_file():
        if str(scene_id) in OPTIONAL_MISSING_SCENE_IDS:
            return None
        raise FileNotFoundError(f"Required scene shard is missing: {path}")
    station_ids, day, data = _read_shard_payload(path)
    return station_ids, day, data


def compile_cluster_year_from_shards(cluster_id, year, samples_year):
    """Choose nearest valid VV/VH acquisitions; exact distance ties prefer earlier dates."""
    cluster_samples = samples_year[samples_year["cluster_id"] == cluster_id].copy()
    if cluster_samples.empty:
        return None

    sample_days = cluster_samples["sample_date"].to_numpy(dtype="datetime64[D]").astype(np.int64)
    min_day, max_day = int(sample_days.min()), int(sample_days.max())
    conn = sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120)
    try:
        scene_rows = conn.execute(
            """SELECT s.scene_id, s.day
               FROM scenes s JOIN scene_clusters sc ON sc.scene_id=s.scene_id
               WHERE sc.cluster_id=? AND s.day>=? AND s.day<=?
               ORDER BY s.day, s.scene_id""",
            (int(cluster_id), min_day - MAX_NEAREST_DAYS, max_day + MAX_NEAREST_DAYS),
        ).fetchall()
    finally:
        conn.close()

    scene_rows = [
        (str(scene_id), int(day)) for scene_id, day in scene_rows
        if not (str(scene_id) in OPTIONAL_MISSING_SCENE_IDS and not shard_path_for_scene(scene_id).is_file())
    ]

    station_observations = {}
    for scene_id, expected_day in scene_rows:
        loaded = load_scene_observation(scene_id)
        if loaded is None:
            continue
        station_ids, stored_day, data = loaded
        if stored_day != expected_day:
            raise RuntimeError(
                f"Scene day mismatch for {scene_id}: SQLite={expected_day}, shard={stored_day}."
            )
        for i, station_id in enumerate(station_ids):
            station_observations.setdefault(str(station_id), []).append((stored_day, scene_id, data[i]))

    for station_id in station_observations:
        station_observations[station_id].sort(key=lambda x: (x[0], x[1]))

    n = len(cluster_samples)
    output = np.full((n, N_OUTPUT_BANDS, PATCH_SIZE, PATCH_SIZE), np.nan, dtype=np.float32)
    observation_counts = np.zeros((n, N_REAL_BANDS), dtype=np.uint8)
    source_day_offset = np.full((n, 2), SOURCE_OFFSET_MISSING, dtype=np.int16)
    station_ids_for_samples = cluster_samples["station_id"].astype(str).to_numpy()

    # HH/HV remain empty because these jobs read only VV and VH.
    pair_definitions = [(0, 1, 0), (2, 3, 1)]
    for row, (station_id, sample_day) in enumerate(zip(station_ids_for_samples, sample_days)):
        entries = station_observations.get(station_id, [])
        if not entries:
            continue
        nearby = [entry for entry in entries if abs(int(entry[0]) - int(sample_day)) <= MAX_NEAREST_DAYS]
        for band_a, band_b, offset_index in pair_definitions:
            candidates = []
            for scene_day, scene_id, data in nearby:
                a, b = data[band_a], data[band_b]
                jointly_valid = np.isfinite(a) & np.isfinite(b)
                n_valid = int(jointly_valid.sum())
                if n_valid < MIN_VALID_PAIR_PIXELS:
                    continue
                delta = int(scene_day) - int(sample_day)
                rank = (abs(delta), 0 if delta <= 0 else 1, -n_valid, str(scene_id))
                candidates.append((rank, scene_day, scene_id, data, n_valid))
            if not candidates:
                continue
            observation_counts[row, band_a] = min(len(candidates), 255)
            observation_counts[row, band_b] = min(len(candidates), 255)
            _, chosen_day, _, chosen_data, _ = min(candidates, key=lambda x: x[0])
            output[row, band_a] = chosen_data[band_a]
            output[row, band_b] = chosen_data[band_b]
            source_day_offset[row, offset_index] = int(chosen_day) - int(sample_day)

    vv = output[:, 0]
    vh = output[:, 1]
    ratio = np.full_like(vv, np.nan)
    valid_ratio = np.isfinite(vv) & np.isfinite(vh) & (vh > 0)
    ratio[valid_ratio] = vv[valid_ratio] / vh[valid_ratio]
    output[:, OUTPUT_BANDS.index("vv_vh_ratio")] = ratio
    return output, observation_counts, source_day_offset


In [7]:
# ============================================================
# YEARLY ZARR STORE CREATION / VALIDATION
# ============================================================
def create_year_zarr(path, n_samples):
    compressor = BloscCodec(cname="zstd", clevel=3, shuffle=BloscShuffle.bitshuffle)
    root = zarr.open_group(path, mode="a")
    if "sample_done" not in root:
        root.attrs.update({
            "schema_version": SCHEMA_VERSION,
            "production_read_strategy": PRODUCTION_READ_STRATEGY,
            "aggregation_method": "COG overview average resampling to 13x13; approximate",
            "collection": S1_COLLECTION,
            "patch_size": PATCH_SIZE,
            "model_resolution_m": MODEL_RESOLUTION_M,
            "source_resolution_m": S1_RESOLUTION_M,
            "max_nearest_days": MAX_NEAREST_DAYS,
            "real_bands": REAL_BANDS,
            "output_bands": OUTPUT_BANDS,
            "temporal_selection": "nearest valid acquisition by absolute date difference; earlier date wins exact ties; VV/VH and HH/HV are selected independently",
            "source_day_offset_columns": ["vv_vh_offset_days", "hh_hv_offset_days"],
            "source_day_offset_missing_value": int(SOURCE_OFFSET_MISSING),
            "vv_vh_ratio": "selected_scene_VV / selected_scene_VH",
            "storage": "native RTC COG range reads + local 10m-to-1km aggregation + nearest-valid scene selection",
        })
        root.create_array(
            "sample_done", shape=(n_samples,),
            chunks=(min(4096, max(1, n_samples)),), dtype="bool", fill_value=False,
        )
        for band in OUTPUT_BANDS:
            root.create_array(
                band, shape=(n_samples, PATCH_SIZE, PATCH_SIZE),
                chunks=(256, PATCH_SIZE, PATCH_SIZE), dtype="float32",
                fill_value=np.nan, compressors=compressor,
            )
        root.create_array(
            "obs_count", shape=(n_samples, N_REAL_BANDS),
            chunks=(1024, N_REAL_BANDS), dtype="uint8", fill_value=0, compressors=None,
        )
        root.create_array(
            "source_day_offset", shape=(n_samples, 2),
            chunks=(1024, 2), dtype="int16", fill_value=SOURCE_OFFSET_MISSING, compressors=None,
        )
    else:
        existing_schema = root.attrs.get("schema_version")
        if existing_schema != SCHEMA_VERSION:
            raise RuntimeError(f"Existing Zarr at {path} has schema_version={existing_schema}; expected {SCHEMA_VERSION}.")
        if root["sample_done"].shape[0] != n_samples:
            raise RuntimeError(f"Existing Zarr has {root['sample_done'].shape[0]} rows; expected {n_samples}.")
        expected_patch_shape = (n_samples, PATCH_SIZE, PATCH_SIZE)
        for band in OUTPUT_BANDS:
            if band not in root or tuple(root[band].shape) != expected_patch_shape:
                raise RuntimeError(f"Existing Zarr band {band!r} is missing or has an incompatible shape.")
        if "obs_count" not in root or tuple(root["obs_count"].shape) != (n_samples, N_REAL_BANDS):
            raise RuntimeError("Existing Zarr obs_count has an incompatible shape.")
        if "source_day_offset" not in root or tuple(root["source_day_offset"].shape) != (n_samples, 2):
            raise RuntimeError("Existing Zarr source_day_offset has an incompatible shape.")
    root.attrs["remote_read_polarizations"] = list(READ_POLARIZATIONS)
    root.attrs["hh_hv_remote_reads_disabled"] = True
    return root


def prepare_year_metadata(year):
    result = (
        samples_c[samples_c["year"] == year]
        .copy()
        .sort_values(["cluster_id", "station_id", "sample_date"], kind="stable")
        .reset_index(drop=True)
    )
    result["row_index"] = np.arange(len(result), dtype=np.int64)
    return result


def run_year(year):
    samples_year = prepare_year_metadata(year)
    if samples_year.empty:
        print(f"Skipping {year}: no sample rows.")
        return None, None

    metadata_path = FINAL_OUTPUT_DIR / f"samples_nearest_valid_{year}.parquet"
    samples_year.to_parquet(metadata_path, index=False)
    zarr_path = FINAL_OUTPUT_DIR / f"s1_nearest_valid_{year}.zarr"
    root = create_year_zarr(zarr_path, len(samples_year))
    done = np.asarray(root["sample_done"][:], dtype=bool)
    cluster_ranges = (
        samples_year.groupby("cluster_id")["row_index"]
        .agg(start="min", end="max").reset_index()
    )
    warned_low_space = False
    for cluster in tqdm(cluster_ranges.itertuples(index=False), total=len(cluster_ranges), desc=f"Year {year}", unit="cluster"):
        cluster_id = int(cluster.cluster_id)
        row_start, row_end = int(cluster.start), int(cluster.end)
        row_slice = slice(row_start, row_end + 1)
        if done[row_slice].all():
            continue

        free_gib = shutil.disk_usage(OUTPUT_DIR).free / (1024 ** 3)
        if free_gib < STOP_FREE_GIB:
            raise RuntimeError(
                f"Paused before year {year}, cluster {cluster_id}: {free_gib:.2f} GiB free; "
                f"the configured minimum is {STOP_FREE_GIB:.1f} GiB. Completed slices are checkpointed; "
                "free disk space and rerun the yearly aggregation cell."
            )
        if free_gib < WARN_FREE_GIB and not warned_low_space:
            print(f"WARNING: only {free_gib:.2f} GiB free. Aggregation will pause below {STOP_FREE_GIB:.1f} GiB.")
            warned_low_space = True

        result = compile_cluster_year_from_shards(cluster_id, year, samples_year)
        if result is None:
            data = np.full((row_end - row_start + 1, N_OUTPUT_BANDS, PATCH_SIZE, PATCH_SIZE), np.nan, dtype=np.float32)
            counts = np.zeros((row_end - row_start + 1, N_REAL_BANDS), dtype=np.uint8)
            offsets = np.full((row_end - row_start + 1, 2), SOURCE_OFFSET_MISSING, dtype=np.int16)
        else:
            data, counts, offsets = result
        expected_rows = row_end - row_start + 1
        if len(data) != expected_rows:
            raise RuntimeError(f"Cluster {cluster_id}: compiler returned {len(data)} rows, expected {expected_rows}.")

        for index, band in enumerate(OUTPUT_BANDS):
            root[band][row_slice] = data[:, index]
        root["obs_count"][row_slice] = counts
        root["source_day_offset"][row_slice] = offsets
        # Mark rows complete only after all arrays have been written.
        root["sample_done"][row_slice] = True
        done[row_slice] = True
        del data, counts, offsets, result
        gc.collect()

    root.attrs["complete"] = bool(np.asarray(root["sample_done"][:], dtype=bool).all())
    print(f"{year}: {int(np.asarray(root['sample_done'][:], dtype=bool).sum()):,}/{len(samples_year):,} samples complete")
    print("Metadata:", metadata_path)
    print("Zarr:", zarr_path)
    return metadata_path, zarr_path


In [8]:
# ============================================================
# RUN YEARLY AGGREGATION (RESUMABLE)
# ============================================================
if not RUN_YEARLY_AGGREGATION:
    print("Yearly aggregation disabled by RUN_YEARLY_AGGREGATION=False.")
else:
    years = sorted(samples_c["year"].dropna().astype(int).unique().tolist())
    print("Starting/resuming yearly aggregation.")
    print("Years:", years[0], "->", years[-1])
    print("Samples:", f"{len(samples_c):,}")
    print("Output directory:", FINAL_OUTPUT_DIR)
    final_results = []
    for year in years:
        print("\n" + "=" * 72)
        print(f"Finalizing {year}")
        metadata_path, zarr_path = run_year(int(year))
        if zarr_path is not None:
            root = zarr.open_group(zarr_path, mode="r")
            complete_rows = int(np.asarray(root["sample_done"][:], dtype=bool).sum())
            total_rows = int(root["sample_done"].shape[0])
            final_results.append({
                "year": int(year),
                "samples": total_rows,
                "completed_samples": complete_rows,
                "complete": bool(root.attrs.get("complete", False)) and complete_rows == total_rows,
                "metadata_path": str(metadata_path),
                "zarr_path": str(zarr_path),
            })
    final_summary = pd.DataFrame(final_results)
    display(final_summary)
    summary_path = FINAL_OUTPUT_DIR / "yearly_aggregation_summary.csv"
    final_summary.to_csv(summary_path, index=False)
    print("Yearly summary:", summary_path)
    if not final_summary.empty and not final_summary["complete"].all():
        print("WARNING: one or more years remain incomplete; rerun this cell to resume remaining slices.")
    elif not final_summary.empty:
        print("All configured sample years have completed yearly Zarr aggregation.")


Starting/resuming yearly aggregation.
Years: 2015 -> 2026
Samples: 3,514,643
Output directory: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1

Finalizing 2015


/tmp/ipykernel_31427/2602867384.py:5: DeprecationWarning: BloscShuffle.bitshuffle is deprecated; pass the string 'bitshuffle' instead.
  compressor = BloscCodec(cname="zstd", clevel=3, shuffle=BloscShuffle.bitshuffle)


Year 2015:   0%|          | 0/665 [00:00<?, ?cluster/s]

2015: 329,903/329,903 samples complete
Metadata: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2015.parquet
Zarr: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1/s1_nearest_valid_2015.zarr

Finalizing 2016


/tmp/ipykernel_31427/2602867384.py:5: DeprecationWarning: BloscShuffle.bitshuffle is deprecated; pass the string 'bitshuffle' instead.
  compressor = BloscCodec(cname="zstd", clevel=3, shuffle=BloscShuffle.bitshuffle)


Year 2016:   0%|          | 0/662 [00:00<?, ?cluster/s]

... (29 outputs left out)

Year 2026:   0%|          | 0/493 [00:00<?, ?cluster/s]

2026: 207,555/207,555 samples complete
Metadata: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1/samples_nearest_valid_2026.parquet
Zarr: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1/s1_nearest_valid_2026.zarr


,year,samples,completed_samples,complete,metadata_path,zarr_path
0,2015,329903,329903,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
1,2016,394790,394790,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
2,2017,369953,369953,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
3,2018,344806,344806,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
4,2019,308679,308679,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
5,2020,269853,269853,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
6,2021,259625,259625,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
7,2022,260490,260490,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
8,2023,233307,233307,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...
9,2024,264946,264946,True,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...,/home/mayur/Code/UG_SM/sentinel1-rtc/Data/Glob...


Yearly summary: /home/mayur/Code/UG_SM/sentinel1-rtc/Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1/yearly_aggregation_summary.csv
All configured sample years have completed yearly Zarr aggregation.
